# 05 — Latent diffusion training (Phase 2)

Train the DiT denoiser on latents from a frozen spectral VAE, using v-prediction and `c_spec` conditioning.

The VAE and prior are frozen; only the DiT is trained.

In [ ]:
import sys
sys.path.insert(0, "../src")

import torch
import matplotlib.pyplot as plt
from ald_sc.build_prior import build_arrow_prior
from ald_sc.vae import SpectralVAE
from ald_sc.dit import MinimalDiT
from ald_sc.schedule import CosineSchedule
from ald_sc.trainer import train_diffusion
from ald_sc.data import ToyImageDataset, build_dataloader

torch.manual_seed(3407)

## 1. Setup: prior, VAE (frozen), DiT, schedule

In [ ]:
F, q = 32, 8
embeddings = torch.randn(64, F)
prior = build_arrow_prior(embeddings, q=q, k=4)

vae = SpectralVAE(in_channels=3, latent_channels=4, feature_dim=F, base_channels=32)
for p in vae.parameters():
    p.requires_grad_(False)
vae.eval()

dit = MinimalDiT(
    latent_channels=4,
    latent_size=8,
    patch_size=2,
    dim=64,
    depth=4,
    num_heads=4,
    text_dim=0,
    spec_dim=3 * q,
    cfg_dropout=0.1,
)

schedule = CosineSchedule(num_steps=1000)

print(f"DiT params: {sum(p.numel() for p in dit.parameters() if p.requires_grad):,}")
print(f"VAE frozen: {not any(p.requires_grad for p in vae.parameters())}")

## 2. Train the DiT

In [ ]:
dataset = ToyImageDataset(num_samples=32, image_size=32, channels=3)
loader = build_dataloader(dataset, batch_size=8)

losses = list(train_diffusion(
    loader, vae, dit, prior, schedule,
    epochs=20, lr=1e-3, cfg_dropout=0.1,
))

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot([l["loss"] for l in losses])
ax.set_xlabel("Step")
ax.set_ylabel("L_diff")
ax.set_title("Diffusion training loss")
ax.set_yscale("log")
plt.tight_layout()
plt.savefig("../results/05_diffusion_loss.png", dpi=150)
plt.show()

## 3. Predicted vs ground-truth velocity

In [ ]:
x = next(iter(loader))
with torch.no_grad():
    z0, A, c_spec, _ = vae(x, prior)

t = torch.tensor([500])
noise = torch.randn_like(z0)
z_t = schedule.add_noise(z0, t, noise)
v_true = schedule.v_target(z0, t, noise)

dit.eval()
with torch.no_grad():
    v_pred = dit(z_t, t, c_spec=c_spec)

error = (v_pred - v_true).pow(2).mean()
print(f"v-prediction MSE at t=500: {error:.6f}")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(z_t[0, 0].numpy(), cmap="viridis")
axes[0].set_title("Noised latent z_t")
axes[1].imshow(v_true[0, 0].numpy(), cmap="RdBu_r", vmin=-1, vmax=1)
axes[1].set_title("True velocity v")
axes[2].imshow(v_pred[0, 0].numpy(), cmap="RdBu_r", vmin=-1, vmax=1)
axes[2].set_title("Predicted velocity v_pred")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.savefig("../results/05_velocity_comparison.png", dpi=150)
plt.show()